## imports & constants

In [44]:
import pandas as pd
import re
import random
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.feature_extraction.text import (
    ENGLISH_STOP_WORDS, CountVectorizer, TfidfVectorizer
)
from sklearn.model_selection import train_test_split

DATA_PATH = './dataset.csv'

## q7

In [18]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def q7_load_and_preprocess():
    df = pd.read_csv(DATA_PATH)
    print("Q7 | Dataset shape:", df.shape)
    df['clean_text'] = df['Text'].apply(preprocess)
    return df

In [19]:
df = q7_load_and_preprocess()
df

Q7 | Dataset shape: (2225, 2)


,Text,Label,clean_text
0,Budget to set scene for election\n \n Gordon B...,0,budget to set scene for election gordon brown ...
1,Army chiefs in regiments decision\n \n Militar...,0,army chiefs in regiments decision military chi...
2,Howard denies split over ID cards\n \n Michael...,0,howard denies split over id cards michael howa...
3,Observers to monitor UK election\n \n Minister...,0,observers to monitor uk election ministers wil...
4,Kilroy names election seat target\n \n Ex-chat...,0,kilroy names election seat target ex chat show...
...,...,...,...
2220,India opens skies to competition\n \n India wi...,4,india opens skies to competition india will al...
2221,Yukos bankruptcy 'not US matter'\n \n Russian ...,4,yukos bankruptcy not us matter russian authori...
2222,Survey confirms property slowdown\n \n Governm...,4,survey confirms property slowdown government f...
2223,High fuel prices hit BA's profits\n \n British...,4,high fuel prices hit ba s profits british airw...


## q8

In [28]:
def q8_top30_words(df, fname, stopwords=False):
    if stopwords:
        stopwords = ENGLISH_STOP_WORDS
    else:
        stopwords = {}
        
    stop_words_set = set(stopwords)

    all_words = []
    for text in df['clean_text']:
        words = [w for w in text.split() if w not in stop_words_set and len(w) > 1]
        all_words.extend(words)

    counter = Counter(all_words)
    top30 = counter.most_common(30)
    print("\nQ8 | Top 30 frequent words:", top30)

    words, counts = zip(*top30)
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.bar(words, counts, color='steelblue')
    plt.xticks(rotation=60, ha='right')
    ax.set_title('Top 30 Most Frequent Words (all documents)')
    ax.set_ylabel('Frequency')
    plt.tight_layout()
    plt.savefig(f'media/{fname}.png', dpi=130)
    plt.close(fig)
    
    return top30

In [29]:
q8_top30_words(df, "q8_top30_wo_stopwords")


Q8 | Top 30 frequent words: [('the', 52636), ('to', 25113), ('of', 20008), ('and', 18612), ('in', 17734), ('for', 8945), ('is', 8555), ('that', 8257), ('it', 7893), ('on', 7625), ('said', 7255), ('was', 6028), ('he', 5939), ('be', 5805), ('with', 5354), ('as', 4981), ('has', 4957), ('have', 4772), ('at', 4638), ('by', 4516), ('will', 4473), ('but', 4421), ('are', 4401), ('from', 3535), ('not', 3484), ('they', 3085), ('his', 3026), ('we', 3006), ('mr', 3005), ('this', 2847)]


[('the', 52636),
 ('to', 25113),
 ('of', 20008),
 ('and', 18612),
 ('in', 17734),
 ('for', 8945),
 ('is', 8555),
 ('that', 8257),
 ('it', 7893),
 ('on', 7625),
 ('said', 7255),
 ('was', 6028),
 ('he', 5939),
 ('be', 5805),
 ('with', 5354),
 ('as', 4981),
 ('has', 4957),
 ('have', 4772),
 ('at', 4638),
 ('by', 4516),
 ('will', 4473),
 ('but', 4421),
 ('are', 4401),
 ('from', 3535),
 ('not', 3484),
 ('they', 3085),
 ('his', 3026),
 ('we', 3006),
 ('mr', 3005),
 ('this', 2847)]

In [30]:
q8_top30_words(df, "q8_top30_w_stopwords", True)


Q8 | Top 30 frequent words: [('said', 7255), ('mr', 3005), ('year', 2310), ('people', 2045), ('new', 1978), ('time', 1322), ('world', 1201), ('government', 1160), ('uk', 1115), ('years', 1003), ('best', 974), ('bn', 958), ('just', 957), ('make', 945), ('told', 911), ('film', 890), ('like', 879), ('game', 871), ('music', 839), ('labour', 804), ('bbc', 778), ('set', 762), ('number', 760), ('way', 740), ('added', 733), ('market', 702), ('says', 687), ('company', 686), ('home', 663), ('election', 662)]


[('said', 7255),
 ('mr', 3005),
 ('year', 2310),
 ('people', 2045),
 ('new', 1978),
 ('time', 1322),
 ('world', 1201),
 ('government', 1160),
 ('uk', 1115),
 ('years', 1003),
 ('best', 974),
 ('bn', 958),
 ('just', 957),
 ('make', 945),
 ('told', 911),
 ('film', 890),
 ('like', 879),
 ('game', 871),
 ('music', 839),
 ('labour', 804),
 ('bbc', 778),
 ('set', 762),
 ('number', 760),
 ('way', 740),
 ('added', 733),
 ('market', 702),
 ('says', 687),
 ('company', 686),
 ('home', 663),
 ('election', 662)]

## q9

In [31]:
def q9_wordcloud(df, fname):
    extra_stop = ['said', 'mr', 'told', 'year', 'years', 'new', 'time', 'best', 'world']
    stop_words = list(ENGLISH_STOP_WORDS.union(extra_stop))

    vectorizer = TfidfVectorizer(stop_words=stop_words, max_df=0.9, min_df=2)
    tfidf = vectorizer.fit_transform(df['clean_text'])
    scores = np.asarray(tfidf.sum(axis=0)).ravel()
    vocab = vectorizer.get_feature_names_out()

    top_idx = np.argsort(scores)[::-1][:20]
    top_words = [(vocab[i], scores[i]) for i in top_idx]

    random.seed(7)
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.axis('off')
    
    max_score = max(s for _, s in top_words)
    min_score = min(s for _, s in top_words)
    placed = []

    def overlaps(x, y, w, h, placed):
        for (px, py, pw, ph) in placed:
            if abs(x - px) < (w + pw) / 2 * 0.75 and abs(y - py) < (h + ph) / 2 * 0.75:
                return True
        return False

    for word, score in top_words:
        size = 16 + 55 * (score - min_score) / (max_score - min_score)
        
        for _ in range(300):
            x = random.uniform(0.1, 0.9)
            y = random.uniform(0.1, 0.9)
            w = len(word) * size * 0.011
            h = size * 0.02
            if not overlaps(x, y, w, h, placed):
                placed.append((x, y, w, h))
                break
        
        color = plt.cm.tab20(random.random())
        ax.text(x, y, word, fontsize=size, ha='center', va='center', color=color,
                fontweight='bold', fontfamily='sans-serif', transform=ax.transAxes)

    ax.set_title('Top 20 Words Wordcloud', fontsize=16)
    plt.tight_layout()
    plt.savefig(f'media/{fname}.png', dpi=130)
    plt.close(fig)
    
    return top_words

In [34]:
q9_wordcloud(df, "q9_wordcloud")

[('people', np.float64(38.11888684148821)),
 ('film', np.float64(32.611031682135625)),
 ('bn', np.float64(30.644135278700738)),
 ('government', np.float64(29.59906836926095)),
 ('uk', np.float64(27.94779443327599)),
 ('labour', np.float64(27.13517911679139)),
 ('game', np.float64(25.897526690640593)),
 ('election', np.float64(23.87570499905268)),
 ('music', np.float64(23.478663030251198)),
 ('blair', np.float64(22.539061919384995)),
 ('england', np.float64(21.92580846308045)),
 ('number', np.float64(21.348826821797545)),
 ('just', np.float64(21.1935742116639)),
 ('make', np.float64(21.02083024538935)),
 ('party', np.float64(20.719867288739334)),
 ('company', np.float64(20.37872593377917)),
 ('market', np.float64(20.310467710648393)),
 ('bbc', np.float64(20.147181031725232)),
 ('like', np.float64(19.879363587634728)),
 ('set', np.float64(19.85851204835371))]

## q10

In [39]:
def q10_bow_matrix(df):
    words_df = pd.read_csv('words.csv', header=None)    
    vocabulary_list = words_df[0].astype(str).str.lower().tolist()
    
    vectorizer = CountVectorizer(vocabulary=vocabulary_list)
    bow = vectorizer.fit_transform(df['clean_text'])
    bow_matrix = bow.toarray()
    
    print(f"\nQ10 | BoW matrix shape: {bow_matrix.shape}  (docs x |vocab|={len(vocabulary_list)})")
    
    return bow_matrix

In [40]:
bow_full = q10_bow_matrix(df)
bow_full


Q10 | BoW matrix shape: (2225, 53)  (docs x |vocab|=53)


array([[0, 0, 1, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(2225, 53))

## Dataset Partitioning

In [42]:
def split_2000_225(df, bow_matrix):    
    """Split literally: first 2000 rows = working set, last 225 = held-out test."""
    df_train = df.iloc[:2000].reset_index(drop=True)
    df_test = df.iloc[2000:].reset_index(drop=True)
    bow_train = bow_matrix[:2000]
    bow_test = bow_matrix[2000:]
    print("Split | train:", df_train.shape, bow_train.shape,
          "| test:", df_test.shape, bow_test.shape)
    return df_train, df_test, bow_train, bow_test

def split_2000_225_shuffle(df, bow_matrix):
    df_train, df_test, bow_train, bow_test = train_test_split(
        df, 
        bow_matrix, 
        train_size=2000, 
        shuffle=True, 
        random_state=42
    )
    
    df_train = df_train.reset_index(drop=True)
    df_test = df_test.reset_index(drop=True)

    print("Split | train:", df_train.shape, bow_train.shape,
          "| test:", df_test.shape, bow_test.shape)
          
    return df_train, df_test, bow_train, bow_test

In [45]:
df_train, df_test, bow_train, bow_test = split_2000_225_shuffle(df, bow_full)

Split | train: (2000, 3) (2000, 53) | test: (225, 3) (225, 53)
